<a href="https://colab.research.google.com/github/ilfpns/HailMary_AI/blob/main/HM_yolo26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import importlib.util
import subprocess
import sys
import os
import shutil

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

import torch
from google.colab import drive
from ultralytics import YOLO
from pathlib import Path
from PIL import Image, ImageDraw
from collections import Counter
from concurrent.futures import ProcessPoolExecutor
from PIL import Image, ImageOps

import pandas as pd
import matplotlib.pyplot as plt

In [8]:
import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

MODULE_DIR = Path("/content/drive/MyDrive/HM_img2")
MODULE = MODULE_DIR / "hm_data.py"

SOURCE = '''import shutil, subprocess
from pathlib import Path

ZIP = Path("/content/drive/MyDrive/HM_img2/HM_img2.zip")
EXTRACT = Path("/content/data")


def prepare_data(zip_path=ZIP, extract=EXTRACT, verbose=True):
    zip_path, extract = Path(zip_path), Path(extract)

    if not any(extract.rglob("data.yaml")):
        assert zip_path.exists(), f"zip이 없습니다: {zip_path}"
        if verbose:
            print("압축 해제:", zip_path.name, f"{zip_path.stat().st_size / 1024**3:.2f} GB")
        local_zip = Path("/content/dataset.zip")
        shutil.copy(zip_path, local_zip)
        extract.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", str(local_zip), "-d", str(extract)], check=True)
        local_zip.unlink()
    elif verbose:
        print("이미 풀려 있음:", extract)

    root = next(extract.rglob("data.yaml")).parent
    yaml_path = root / "data.yaml"
    lines = [l for l in yaml_path.read_text().splitlines() if not l.startswith("path:")]
    yaml_path.write_text("\\n".join([f"path: {root}"] + lines) + "\\n")

    if verbose:
        print("root:", root)
        for s in ["train", "valid", "test"]:
            if (root / s).exists():
                n_img = len(list((root / s / "images").iterdir()))
                n_lbl = len(list((root / s / "labels").iterdir()))
                print(f"{s:<6} 이미지 {n_img}  라벨 {n_lbl}")

    return root, yaml_path
'''

if MODULE.exists():
    print("모듈 있음:", MODULE)
else:
    MODULE_DIR.mkdir(parents=True, exist_ok=True)
    MODULE.write_text(SOURCE, encoding="utf-8")
    print("모듈 생성:", MODULE)

if str(MODULE_DIR) not in sys.path:
    sys.path.append(str(MODULE_DIR))
from hm_data import prepare_data

root, yaml_path = prepare_data()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
모듈 있음: /content/drive/MyDrive/HM_img2/hm_data.py
이미 풀려 있음: /content/data
root: /content/data/HM_img2
train  이미지 20885  라벨 20885
valid  이미지 1531  라벨 1531
test   이미지 356  라벨 356


In [9]:
MAX = 640
root = next(Path("/content/data").rglob("data.yaml")).parent
paths = [p for s in ["train", "valid", "test"] for p in (root / s / "images").iterdir()]


def shrink(p):
    with Image.open(p) as im:
        size = im.size
        if max(size) <= MAX:
            return None
        im.draft("RGB", (MAX, MAX))
        out = ImageOps.exif_transpose(im).convert("RGB")
    out.thumbnail((MAX, MAX), Image.LANCZOS)
    out.save(p, quality=95)
    return f"{p.parent.parent.name}\t{p.stem}\t{size[0]}\t{size[1]}"


with ProcessPoolExecutor(max_workers=os.cpu_count()) as ex:
    large = [r for r in ex.map(shrink, paths, chunksize=64) if r]
print(f"줄인 이미지: {len(large)} / {len(paths)}")

Path("/content/drive/MyDrive/HM_img2/large_images.tsv").write_text("split\tstem\tw\th\n" + "\n".join(large) + "\n")

for c in root.rglob("*.cache"):
    c.unlink()

subprocess.run("cd /content/data && zip -qr -0 /content/HM_img2_640.zip .", shell=True, check=True)
shutil.copy("/content/HM_img2_640.zip", "/content/drive/MyDrive/HM_img2/HM_img2_640.zip")
print("백업 완료:", round(os.path.getsize("/content/HM_img2_640.zip") / 1024**3, 2), "GB")

줄인 이미지: 5378 / 22772
백업 완료: 1.79 GB


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.4"], check=True)

assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 → 런타임 유형 변경 → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/HM_img2")
ZIP = BASE / "HM_img2_640.zip"
MODULE = BASE / "hm_data.py"
PROJECT = BASE / "runs"
NAME = "HM_yolo26n"
MODEL = "yolo26n.pt"
EPOCHS = 100

SOURCE = '''import shutil, subprocess
from pathlib import Path

ZIP = Path("/content/drive/MyDrive/HM_img2/HM_img2_640.zip")
EXTRACT = Path("/content/data")


def prepare_data(zip_path=ZIP, extract=EXTRACT, verbose=True):
    zip_path, extract = Path(zip_path), Path(extract)

    if not any(extract.rglob("data.yaml")):
        assert zip_path.exists(), f"zip이 없습니다: {zip_path}"
        if verbose:
            print("압축 해제:", zip_path.name, f"{zip_path.stat().st_size / 1024**3:.2f} GB")
        local_zip = Path("/content/dataset.zip")
        shutil.copy(zip_path, local_zip)
        extract.mkdir(parents=True, exist_ok=True)
        subprocess.run(["unzip", "-q", "-o", str(local_zip), "-d", str(extract)], check=True)
        local_zip.unlink()
    elif verbose:
        print("이미 풀려 있음:", extract)

    root = next(extract.rglob("data.yaml")).parent
    yaml_path = root / "data.yaml"
    lines = [l for l in yaml_path.read_text().splitlines() if not l.startswith("path:")]
    yaml_path.write_text("\\n".join([f"path: {root}"] + lines) + "\\n")

    if verbose:
        print("root:", root)
        for s in ["train", "valid", "test"]:
            if (root / s).exists():
                n_img = len(list((root / s / "images").iterdir()))
                n_lbl = len(list((root / s / "labels").iterdir()))
                print(f"{s:<6} 이미지 {n_img}  라벨 {n_lbl}")

    return root, yaml_path
'''

if not MODULE.exists():
    BASE.mkdir(parents=True, exist_ok=True)
    MODULE.write_text(SOURCE, encoding="utf-8")
    print("모듈 생성:", MODULE)

if str(BASE) not in sys.path:
    sys.path.append(str(BASE))
from hm_data import prepare_data

root, yaml_path = prepare_data(ZIP)

weights = PROJECT / NAME / "weights"


def load_epoch(pt):
    try:
        return torch.load(pt, map_location="cpu", weights_only=False).get("epoch", -1)
    except Exception as e:
        print("체크포인트 읽기 실패:", pt.name, e)
        return None


def find_checkpoint():
    if not weights.exists():
        return None, None
    periodic = sorted(
        weights.glob("epoch*.pt"),
        key=lambda p: int(re.findall(r"\d+", p.stem)[0]),
        reverse=True,
    )
    for pt in [weights / "last.pt", *periodic]:
        if pt.exists():
            ep = load_epoch(pt)
            if ep is not None:
                return pt, ep
    return None, None


ckpt, ep = find_checkpoint()

if ckpt is not None and ep == -1:
    print("이미 학습이 끝난 run입니다:", PROJECT / NAME)
elif ckpt is not None:
    print(f"이어서 학습: {ckpt.name} (완료된 epoch {ep + 1}/{EPOCHS})")
    YOLO(str(ckpt)).train(resume=True)
else:
    print("처음부터 학습")
    YOLO(MODEL).train(
        data=str(yaml_path),
        epochs=EPOCHS,
        patience=30,
        imgsz=640,
        batch=32,
        workers=2,
        device=0,
        project=str(PROJECT),
        name=NAME,
        exist_ok=True,
        save_period=2,
        seed=42,
    )

best = YOLO(str(weights / "best.pt"))
m = best.val(data=str(yaml_path), split="test", imgsz=640)
for j, c in enumerate(m.box.ap_class_index):
    print(f"{best.names[int(c)]:<12} mAP50={m.box.ap50[j]:.3f}  mAP50-95={m.box.ap[j]:.3f}")

GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
이미 풀려 있음: /content/data
root: /content/data/HM_img2
train  이미지 20885  라벨 20885
valid  이미지 1531  라벨 1531
test   이미지 356  라벨 356
이어서 학습: last.pt (완료된 epoch 6/100)
New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/HM_img2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=F